<a href="https://colab.research.google.com/github/Irosha-Ranaweera/ai-image-detection-generalization/blob/main/notebooks/02_colab_full_experiment_baseline_vs_eca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 02 - Full Experiment in Colab

This notebook runs the full comparison for AI-generated image detection:

1. Baseline ResNet18
2. ResNet18 + Efficient Channel Attention (ECA)

The dataset must have this structure:

```text
ddata/
  train/
    fake/
    real/
  val/
    fake/
    real/
  test/
    fake/
    real/
```

The metrics in this project treat `fake` as the positive class.

## 1. Select GPU

In Colab, choose:

`Runtime > Change runtime type > GPU`

Then run the cell below.

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Change runtime type to GPU before full training.")

CUDA available: False
Running on CPU. Change runtime type to GPU before full training.


## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Go to the project folder

Update `PROJECT_DIR` if your repository is stored in a different Google Drive folder.

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/ai-image-detection-generalization"

%cd "$PROJECT_DIR"
!pwd
!ls

## 4. Install requirements

In [ ]:
!pip install -q torch torchvision scikit-learn matplotlib tqdm seaborn

## 5. Set full dataset path

Your Windows path:

```text
G:\My Drive\Master of AI ML\Semester01-2026\Deep learning\Deep learning Assignment\datasets\ddata
```

becomes this Colab path:

```text
/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/datasets/ddata
```

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/datasets/ddata"

!ls "$DATA_DIR"
!ls "$DATA_DIR/train"
!ls "$DATA_DIR/val"
!ls "$DATA_DIR/test"

## 6. Count dataset images

This confirms the number of images in each split and class before training.

In [ ]:
from pathlib import Path

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
data_path = Path(DATA_DIR)

for split in ["train", "val", "test"]:
    for class_name in ["fake", "real"]:
        folder = data_path / split / class_name
        count = sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in image_extensions)
        print(f"{split}/{class_name}: {count}")

## 7. Common experiment settings

Use the same settings for both models so the comparison is fair.

In [ ]:
import os

os.environ["DATA_DIR"] = DATA_DIR
os.environ["MODEL_NAME"] = "resnet18"
os.environ["EPOCHS"] = "10"
os.environ["BATCH_SIZE"] = "32"
os.environ["LEARNING_RATE"] = "1e-4"
os.environ["NUM_WORKERS"] = "2"
os.environ["OUTPUT_DIR"] = "outputs/figures"

print("DATA_DIR:", os.environ["DATA_DIR"])
print("MODEL_NAME:", os.environ["MODEL_NAME"])
print("EPOCHS:", os.environ["EPOCHS"])
print("BATCH_SIZE:", os.environ["BATCH_SIZE"])
print("LEARNING_RATE:", os.environ["LEARNING_RATE"])
print("NUM_WORKERS:", os.environ["NUM_WORKERS"])
print("OUTPUT_DIR:", os.environ["OUTPUT_DIR"])

## 8. Train baseline ResNet18

This trains the baseline model and evaluates it on the test set.

Saved model:

```text
best_resnet18.pth
```

In [ ]:
# Dataset copied locally for speed
!cp -r "$DATA_DIR" /content/ddata
os.environ["DATA_DIR"] = "/content/ddata"



In [ ]:
os.environ.pop("CHECKPOINT_PATH", None)
# os.environ["SAVE_PATH"] = "outputs/checkpoints/best_resnet18.pth"
RESULTS_DIR = "/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/results"

# Results saved to Drive for safety
os.environ["SAVE_PATH"] = f"{RESULTS_DIR}/checkpoints/best_resnet18.pth"
os.environ["OUTPUT_DIR"] = f"{RESULTS_DIR}/figures"


!python scripts/train_baseline.py

Using device: cuda
Classes: ['fake', 'real']

Epoch 1/10
Training:  87% 1974/2261 [2:34:39<20:45,  4.34s/it]

### Run from checkpoint

In [ ]:
os.environ["CHECKPOINT_PATH"] =f"{RESULTS_DIR}/checkpoints/best_resnet18.pth"
os.environ["SAVE_PATH"] =f"{RESULTS_DIR}/checkpoints/best_resnet18_continued.pth"

!python scripts/train_baseline.py


## 9. Train ResNet18 + ECA

This trains the attention-enhanced model using the same dataset and hyperparameters.

Saved model:

```text
best_eca_resnet18.pth
```

In [ ]:
os.environ.pop("CHECKPOINT_PATH", None)
os.environ["SAVE_PATH"] = "outputs/checkpoints/best_eca_resnet18.pth"

!python scripts/train_attention.py

## 10. Record results for the report

After both scripts finish, copy the final test metrics into a table like this:

| Model | Accuracy | Precision(fake) | Recall(fake) | F1(fake) |
|---|---:|---:|---:|---:|
| ResNet18 baseline |  |  |  |  |
| ResNet18 + ECA |  |  |  |  |

Use the confusion matrices to explain which model makes fewer false positives and false negatives.

## 11. Interpretation guide

- `Accuracy`: total correct predictions across fake and real images.
- `Precision(fake)`: when the model predicts fake, how often it is correct.
- `Recall(fake)`: how many fake images the model successfully detects.
- `F1(fake)`: balance between fake precision and fake recall.

For this assignment, `Recall(fake)` and `F1(fake)` are especially important because the task is AI-generated image detection.